# Ranking-evaluation queries

Issue: #10

## Question

OTTO has no search queries. What can we use as a stand-in so we can evaluate ranking?

## Data used

Inputs: `data/sample/events.parquet` (#4), the evaluation part of window w0 (#6, days 14-15), and `data/categories/item_category.parquet` (#8). The query category is a cluster ID for now; readable names come from #9 and can be joined later.

Output: `data/eval/ranking_queries.parquet`. Code: `notebooks/queries.py` (`from queries import build_ranking_queries`).

Columns: `session`, `prefix_aid`/`prefix_type` (events before the cut), `prefix_ts_last`, `query_category`, `label_aid`/`label_type` (held-out events) and `label_cluster` (the category of each label item, -1 if it has none), `label_ts_first`.

## Known leak

**The query category comes from the label.** We take the category of the held-out next click as the query. That means the query already contains part of the answer, so ranking inside the category is **easier than real search**, where the system has to work out the intent from text. It is a stand-in for intent, used because OTTO has no query text. Every number computed on these queries must be reported with this caveat. The classifier (#34 to #36) is tested separately, and #36 measures what happens when the category is predicted instead of taken from the label.

## How a query is made

1. Take an evaluation session (it starts inside the two evaluation days).
2. Choose a cut position at random among the session's clicks (not the first event) whose item has a category.
3. The events before the cut are the **prefix**. The category of the click at the cut is the **query category**.
4. **Labels** are the events from the cut on, up to a horizon of 10 events: the held-out click plus what follows (clicks, carts, orders).

Sessions with fewer than 2 events, or with no click on a categorised item after the first event, are excluded and counted. At most 200,000 queries are kept, stratified by prefix length (not needed here, since there are fewer sessions than that).

In [ ]:
import sys
from pathlib import Path

import polars as pl

sys.path.insert(0, ".")
sys.path.insert(0, "notebooks")
from queries import build_ranking_queries, prefix_bucket
from split import split_window

DATA = Path("data") if Path("data").exists() else Path("../data")
OUT = DATA / "eval"
OUT.mkdir(parents=True, exist_ok=True)

## A toy check first

One session with events (item, type): 1 click, 2 click, 3 cart, 4 click, 5 order. Items 2 and 4 are in categories 7 and 8, item 1 in category 7, others none.

Eligible cut positions are clicks after the first event on a categorised item: positions 1 (item 2, category 7) and 3 (item 4, category 8). Whichever is drawn, the checks below must hold.

In [ ]:
toy = pl.DataFrame({
    "session": [1] * 5,
    "aid": [1, 2, 3, 4, 5],
    "ts": [10, 20, 30, 40, 50],
    "type": [0, 0, 1, 0, 2],
})
toy_q, toy_stats = build_ranking_queries(toy, {1: 7, 2: 7, 4: 8}, seed=0)
row = toy_q.row(0, named=True)
assert toy_q.height == 1
assert row["query_category"] in (7, 8)
if row["query_category"] == 7:      # cut at position 1
    assert row["prefix_aid"] == [1] and row["label_aid"] == [2, 3, 4, 5]
    assert row["label_cluster"] == [7, -1, 8, -1]      # items without category are flagged -1
else:                                # cut at position 3
    assert row["prefix_aid"] == [1, 2, 3] and row["label_aid"] == [4, 5]
assert row["label_ts_first"] > row["prefix_ts_last"]
# a session whose only later click has no category is excluded
_, s2 = build_ranking_queries(toy.head(3), {1: 7}, seed=0)
assert s2["no_categorised_click"] == 1 and s2["usable"] == 0
print("toy checks pass:", row["prefix_aid"], "->", row["label_aid"], "category", row["query_category"])

## Build for the real evaluation window (w0)

In [ ]:
categories = pl.read_parquet(DATA / "categories" / "item_category.parquet")
item_cluster = dict(zip(categories["aid"].to_list(), categories["cluster"].to_list()))

_, evaluation = split_window(pl.scan_parquet(DATA / "sample" / "events.parquet"), "w0 initial (weeks 1-2)")
eval_events = evaluation.select("session", "aid", "ts", "type").sort("session", "ts").collect()
print(f"evaluation events: {eval_events.height:,}, sessions: {eval_events['session'].n_unique():,}")

queries, stats = build_ranking_queries(eval_events, item_cluster, cap=200_000, seed=0)
print(stats)
queries.write_parquet(OUT / "ranking_queries.parquet")
print(queries.head(3))

## Checks

In [ ]:
q = pl.read_parquet(OUT / "ranking_queries.parquet")
expected = ["session", "prefix_aid", "prefix_type", "prefix_ts_last", "query_category",
            "label_aid", "label_type", "label_cluster", "label_ts_first"]
assert q.columns == expected
assert q["session"].n_unique() == q.height
assert (q["prefix_aid"].list.len() >= 1).all(), "empty prefix"
assert (q["label_aid"].list.len() >= 1).all(), "no labels"
assert (q["label_ts_first"] >= q["prefix_ts_last"]).all(), "a label is before the prefix end"
ties = (q["label_ts_first"] == q["prefix_ts_last"]).sum()
print(f"{q.height:,} queries; labels never precede the prefix; {ties:,} ({ties / q.height:.1%}) have the first label at the same millisecond as the last prefix event")

# the query category is the category of the first label (the held-out click), by construction
first_cluster = q["label_cluster"].list.first()
assert (first_cluster == q["query_category"]).all()
assert (q["label_type"].list.first() == 0).all()  # the held-out event is a click
print("query category equals the category of the held-out click for every query")

## What the queries look like

In [ ]:
share = {k: stats[k] for k in ("eval_sessions", "too_short", "no_categorised_click", "usable", "kept")}
print(pl.DataFrame({"count": list(share.values())}, ).with_columns(pl.Series("what", list(share))).select("what", "count"))

top = q.group_by("query_category").len().sort("len", descending=True)
print("categories with queries:", top.height, "of", categories["cluster"].n_unique())
print(top.head(10))
print(top.select(pl.col("len").min().alias("min"), pl.col("len").median().alias("median"), pl.col("len").max().alias("max")))

buckets = pl.DataFrame({"prefix_length": [prefix_bucket(n) for n in q["prefix_aid"].list.len().to_list()]})
print(buckets.group_by("prefix_length").len().sort("prefix_length").with_columns((pl.col("len") / q.height).alias("share")))

label_len = q["label_aid"].list.len()
print("labels per query: median", label_len.median(), "mean", round(label_len.mean(), 2))
in_cat = q.select(pl.col("label_cluster").list.eval(pl.element() >= 0).list.sum()).to_series().sum()
print(f"label events on items with a category: {in_cat:,} of {label_len.sum():,}")

## The cap works and keeps prefix-length shares (not triggered above)

In [ ]:
capped, _ = build_ranking_queries(eval_events, item_cluster, cap=10_000, seed=0)
assert abs(capped.height - 10_000) <= 4
full = pl.DataFrame({"b": [prefix_bucket(n) for n in q["prefix_aid"].list.len().to_list()]}).group_by("b").len().sort("b")
part = pl.DataFrame({"b": [prefix_bucket(n) for n in capped["prefix_aid"].list.len().to_list()]}).group_by("b").len().sort("b")
for (b, n_full), (_, n_part) in zip(full.iter_rows(), part.iter_rows(), strict=True):
    assert abs(n_full / q.height - n_part / capped.height) < 0.005, b
print(f"cap=10,000 keeps {capped.height:,} queries with the same prefix-length shares")

## Result

The file has one query per usable evaluation session, built from window w0's two evaluation days, with the known leak stated above. Counts of excluded sessions are in the stats printout.

## What I learned

To be written by the owner of the project.